[⬅ Back to the examples](https://s-celles.github.io/anywidget-instruments/try/)

# SVG faceplates

Every panel below is an SVG drawing, as a designer would draw it in a
vector editor. Elements labelled `awi:<role>=<name>` are animated from
Python (a needle turns, a level rises, a lamp lights) or act as controls
(a position of the selector, the FILL and DRAIN push buttons); the rest is
decoration. The five drawings are the templates shipped with the library.

**Run all the cells**, then turn the **selector** to HAND and hold **FILL** or **DRAIN**; in AUTO the
pump keeps the tank between 20 % and 80 % while the process draws water.
The level transmitter sends 0 to 10 V to the **voltmeter**; the
**pressure gauge** shows the pump discharge pressure.

> **The code is hidden.** To see a cell's code, click the `⋯` bar above its result; to see all of it, choose **View ▸ Expand All Code**.

In [ ]:
# JupyterLite: install the package (bundled with this site) in the browser kernel
%pip install -q anywidget-instruments

In [ ]:
import anywidget_instruments as ai

ai.theme_switch()  # light / system / dark theme (STYLE-008)

In [ ]:
import asyncio

tank = ai.SvgPanel.template("tank", label="Tank T-101", size=(220, 250))
selector = ai.SvgPanel.template("selector", label="Pump mode", size=(190, 210))
lamp = ai.SvgPanel.template("pilot_lamp", mode="indicator", label="Pump", size=(140, 190))
voltmeter = ai.SvgPanel.template("voltmeter", mode="indicator", label="LT-101 signal")
gauge = ai.SvgPanel.template(
    "pressure_gauge", mode="indicator", label="Discharge pressure", size=(210, 230)
)
tank.update(value=50.0, high=False, fill=False, drain=False)
selector.update(value=1, mode="HAND")
lamp.update(value=False, caption="Pump P-101")
MODES = {0: "OFF", 1: "HAND", 2: "AUTO"}
sim = {"auto_fill": False}


def tick():
    """Advance the process by one second."""
    mode = MODES.get(int(selector["value"] or 0), "OFF")
    level = float(tank["value"] or 0.0)
    if mode == "HAND":
        pumping = bool(tank["fill"])
        level += (6.0 if pumping else 0.0) - (6.0 if tank["drain"] else 0.0)
    elif mode == "AUTO":
        if level < 20:
            sim["auto_fill"] = True
        elif level > 80:
            sim["auto_fill"] = False
        pumping = sim["auto_fill"]
        level += (8.0 if pumping else 0.0) - 3.0
    else:
        pumping = False
    level = min(100.0, max(0.0, level))
    with ai.batch():
        tank.update(value=level, high=level > 85)
        selector["mode"] = mode
        lamp["value"] = pumping
        voltmeter["value"] = level / 10.0  # 0-10 V transmitter signal
        gauge["value"] = 5.5 + level * 0.02 if pumping else 0.2


async def run(ticks=1800):
    for _ in range(ticks):
        tick()
        await asyncio.sleep(1.0)


task = asyncio.ensure_future(run())  # noqa: RUF006 - stop it with task.cancel()
ai.Panel([tank, selector, lamp, voltmeter, gauge], columns=3)